# 01 · Caracterización del corpus

**Objetivo específico 1.** Caracterizar el corpus de reseñas mediante análisis
descriptivo y verificación de calidad, para definir los requerimientos de su
preparación.

El notebook describe qué hay en el corpus —composición, calidad, distribución por
calificación, establecimiento y zona— y deja explícitos los problemas que condicionan
todo lo que viene después.

In [ ]:
# Configuración local. El kernel es el del entorno de uv:
#     uv run python -m ipykernel install --user --name topicos-popayan
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
print(f"raíz del proyecto: {RAIZ}")

In [ ]:
import pandas as pd

from src.config import DIMENSIONES, DATOS_PROCESADOS, FIGURAS

pd.set_option("display.max_colwidth", 95)

fragmentos = pd.read_csv(DATOS_PROCESADOS / "fragmentos.csv")
resenas = fragmentos.groupby("review_id").agg(
    establecimiento=("establecimiento", "first"),
    zona=("zona", "first"),
    rating=("rating", "first"),
    n_fragmentos=("fragmento_id", "size"),
    largo=("largo_caracteres", "sum"),
)
print(f"{len(fragmentos)} fragmentos · {len(resenas)} reseñas · "
      f"{fragmentos['place_id'].nunique()} establecimientos")

## 1. Composición

El corpus es de reseñas de Google Maps de establecimientos gastronómicos de Popayán,
en español y con al menos 50 caracteres. La unidad de análisis es el fragmento.

In [ ]:
composicion = pd.Series({
    "Reseñas": len(resenas),
    "Fragmentos": len(fragmentos),
    "Establecimientos": fragmentos["place_id"].nunique(),
    "Zonas": fragmentos["zona"].nunique(),
    "Fragmentos por reseña (mediana)": resenas["n_fragmentos"].median(),
    "Fragmentos por reseña (máximo)": resenas["n_fragmentos"].max(),
}, name="valor").to_frame()
composicion

## 2. Longitud de las unidades

El fragmento mediano tiene 10 palabras. Es una unidad corta, muy por debajo del truncado
de cualquier modelo de embeddings, pero también **demasiado corta para que dos términos
de un mismo tópico coaparezcan en ella**, lo que penaliza las medidas de coherencia por
co-ocurrencia que se calculan en el notebook 03.

In [ ]:
largos = pd.DataFrame({
    "caracteres": fragmentos["largo_caracteres"],
    "palabras": fragmentos["largo_palabras"],
}).describe(percentiles=[.25, .5, .75, .9]).round(1)
largos

## 3. Verificación de calidad

Cuatro comprobaciones sobre el corpus que se hereda.

In [ ]:
nulos = fragmentos.isna().sum()
print("columnas con nulos:")
print(nulos[nulos > 0].to_string() if nulos.any() else "  ninguna")

vacios = int((fragmentos["texto_limpio"].fillna("").str.len() == 0).sum())
print(f"\ntexto_limpio vacío tras las stopwords: {vacios} "
      f"({100 * vacios / len(fragmentos):.1f} %) — se conservan, porque su `texto` sí sirve")
print(f"texto duplicado exacto entre fragmentos: {fragmentos['texto'].duplicated().sum()}")

### Una fuga en la detección de idioma heredada

Un fragmento está escrito en alfabeto árabe y pasó el filtro como apto en español. Es
1 de 5.913 (0,02 %) y quedó sin asignar a ningún tópico, así que no afecta ningún
resultado. Se deja constancia porque indica que la selección del corpus, que se reutiliza
por decisión, no es perfecta.

In [ ]:
import unicodedata

# La comprobación va en Python y no en `str.contains`: el motor de regex de pandas 3
# es RE2 vía Arrow, que no admite escapes \uXXXX
ESCRITURAS_LATINAS = ("LATIN", "COMMON", "INHERITED")

def es_no_latino(texto):
    return any(
        not unicodedata.name(c, "").startswith(ESCRITURAS_LATINAS) and c.isalpha()
        for c in str(texto)
    )

fuga = fragmentos[fragmentos["texto"].map(es_no_latino)]
print(f"fragmentos con escritura no latina: {len(fuga)}")
fuga[["fragmento_id", "texto", "rating"]]

## 4. Distribución por calificación

El corpus está **fuertemente sesgado hacia los extremos**, que es el patrón habitual de
las reseñas en línea: se escribe cuando se está muy satisfecho o muy molesto.

In [ ]:
por_rating = resenas["rating"].value_counts().sort_index().to_frame("reseñas")
por_rating["% reseñas"] = (100 * por_rating["reseñas"] / len(resenas)).round(1)
por_rating["fragmentos"] = fragmentos["rating"].value_counts().sort_index()
por_rating["% fragmentos"] = (100 * por_rating["fragmentos"] / len(fragmentos)).round(1)
por_rating.index.name = "estrellas"
print(por_rating.to_string())
print(f"\nrating medio: reseña {resenas['rating'].mean():.2f} · "
      f"fragmento {fragmentos['rating'].mean():.2f}")

**Cada fragmento hereda la calificación de su reseña**, y esa es una limitación
estructural del diseño: la calificación es una sola por reseña y se replica en sus 2,4
fragmentos promedio. De ahí que el rating medio por fragmento difiera del de reseña —las
reseñas largas pesan más— y que los fragmentos de una misma reseña no sean
observaciones independientes.

In [ ]:
comparacion = pd.DataFrame({
    "media de fragmentos por reseña": resenas.groupby("rating")["n_fragmentos"].mean().round(2),
    "reseñas": resenas["rating"].value_counts().sort_index(),
})
comparacion.index.name = "estrellas"
comparacion

Las reseñas de 1 estrella son las más largas: quien se queja explica. Eso las
sobrerrepresenta en el corpus de fragmentos respecto al de reseñas, y hay que tenerlo
presente al leer cualquier porcentaje calculado sobre fragmentos.

## 5. Distribución por establecimiento

Importa porque un corpus concentrado en pocos locales produciría tópicos que en realidad
describen un restaurante, no un tema.

In [ ]:
por_local = fragmentos["establecimiento"].value_counts()
resumen = pd.Series({
    "Establecimientos con fragmentos": len(por_local),
    "Fragmentos del local mayor": por_local.iloc[0],
    "% del corpus en el local mayor": round(100 * por_local.iloc[0] / len(fragmentos), 1),
    "% del corpus en los 10 mayores": round(100 * por_local.head(10).sum() / len(fragmentos), 1),
    "Mediana de fragmentos por local": int(por_local.median()),
}, name="valor").to_frame()
print(resumen.to_string())
por_local.head(8).to_frame("fragmentos")

## 6. Distribución por zona

La zona viene del marco muestral y distingue si el establecimiento está dentro o fuera
del centro histórico, que es la variable con carga patrimonial del estudio.

In [ ]:
por_zona = fragmentos.groupby("zona").agg(
    fragmentos=("fragmento_id", "size"),
    resenas=("review_id", "nunique"),
    locales=("place_id", "nunique"),
    rating=("rating", "mean"),
).round(2)
por_zona["% fragmentos"] = (100 * por_zona["fragmentos"] / len(fragmentos)).round(1)
por_zona["% patrimonio"] = (100 * fragmentos.groupby("zona")["patrimonio"].mean()).round(1)
por_zona.sort_values("fragmentos", ascending=False)

## 7. Prevalencia de las seis dimensiones

In [ ]:
prevalencia = pd.DataFrame({
    "fragmentos": [int(fragmentos[d].sum()) for d in DIMENSIONES],
    "% fragmentos": [round(100 * fragmentos[d].mean(), 1) for d in DIMENSIONES],
    "% reseñas": [round(100 * fragmentos.groupby("review_id")[d].any().mean(), 1)
                  for d in DIMENSIONES],
    "rating medio": [round(fragmentos.loc[fragmentos[d], "rating"].mean(), 2)
                     for d in DIMENSIONES],
}, index=list(DIMENSIONES))
prevalencia.sort_values("fragmentos", ascending=False)

## 8. El embudo del corpus

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(FIGURAS / "04_embudo_corpus.png"), width=900))

## Hallazgos

1. **El corpus es pequeño y desbalanceado hacia los extremos.** 2.430 reseñas y 5.913
   fragmentos de unos 120 establecimientos, con una distribución de calificaciones en U
   típica de las reseñas en línea. Cualquier media hay que leerla sabiendo que la mezcla
   no es la de la población de comensales, sino la de quienes deciden escribir.

2. **Las reseñas negativas son más largas.** Las de 1 estrella aportan más fragmentos
   por reseña que las de 5, lo que las sobrerrepresenta en el corpus de fragmentos.
   Es el motivo por el que el rating medio del fragmento difiere del de la reseña.

3. **La concentración por establecimiento es moderada** y no amenaza la validez del
   modelado: ningún local domina el corpus.

4. **El fragmento mediano tiene 10 palabras.** Es bueno para los embeddings, pero
   demasiado corto para que dos términos de un tópico coaparezcan, lo que condiciona las
   medidas de coherencia del notebook 03 y hay que advertirlo antes de interpretarlas.

5. **Requerimientos de preparación que se derivan de aquí:** segmentar por cláusula para
   ganar pureza temática, enmascarar los nombres de establecimiento para que el modelo no
   agrupe por local, conservar las negaciones en el texto de c-TF-IDF, y no aplicar
   pruebas que asuman independencia entre fragmentos.

---

*Este notebook lee de `data/processed/` los resultados ya calculados. Los scripts que
los producen están en `scripts/` y las decisiones que los sustentan, con sus cifras, en
`docs/bitacora.md` y `docs/decisiones_metodologicas.md`.*